# Lesson 01 — A single node (the baseline)

This is the shape the project started with: `START -> extract -> END`.

It works, but notice what it *doesn't* exercise. One node with a straight line
through it is just a function call wrapped in graph ceremony — no branching, no
loops, no state carried across steps. Every later lesson adds exactly one
LangGraph mechanism on top of this same extraction task.

**Setup:** the reusable pieces (schema, model, prompt, data, renderer) now live in
the `clinical` package, installed with `pip install -e .`. No more per-notebook
`%pip install`.

In [ ]:
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import START, END, StateGraph

from clinical import (
    EXTRACTION_SYSTEM_PROMPT,
    SymptomExtraction,
    build_extraction_user_message,
    get_extraction_model,
    sample_transcript,
    show_symptom_evidence_matrix,
)

In [ ]:
class SymptomNoteState(TypedDict, total=False):
    transcript: str
    encounter_date: str
    extraction: SymptomExtraction


model = get_extraction_model()


def extract_clinical_data(state: SymptomNoteState) -> dict:
    messages = [
        SystemMessage(content=EXTRACTION_SYSTEM_PROMPT),
        HumanMessage(content=build_extraction_user_message(
            state["transcript"], state.get("encounter_date", "not provided"))),
    ]
    return {"extraction": model.invoke(messages)}


builder = StateGraph(SymptomNoteState)
builder.add_node("extract_clinical_data", extract_clinical_data)
builder.add_edge(START, "extract_clinical_data")
builder.add_edge("extract_clinical_data", END)
graph = builder.compile()

In [ ]:
# The whole graph, as a picture. It's a straight line — that's the point.
print(graph.get_graph().draw_mermaid())

In [ ]:
# Runs the real model via OpenRouter (needs OPENROUTER_API_KEY in .env; costs a call).
result = graph.invoke({
    "transcript": sample_transcript(seed=7),
    "encounter_date": "27.11.2025",
})
show_symptom_evidence_matrix(result["extraction"])

## What this taught — and what's missing

You built and ran a LangGraph graph, but a linear single-node graph can't:

- react to its own output (what if the model cites evidence that isn't in the transcript?),
- accumulate state across steps,
- or loop.

➡️ **Lesson 04** turns this into a real graph: `extract -> validate -> (loop back)`.